# HartmannDeveloping — developing (inertial) Hartmann flow  `MagneticCoupled`


In [ ]:
from trustutils import run

run.TRUST_parameters("1.9.8")
run.reset()


In [ ]:
import numpy as np
import os
from trustutils.files import SonFile

BUILD = run.BUILD_DIRECTORY

def probe_xy(path):
    """Effective (x, y) of a point probe, from the .son header."""
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.startswith("# Temps"):
                toks = line.replace("=", " = ").split()
                return float(toks[toks.index("x") + 2]), float(toks[toks.index("y") + 2])
    raise RuntimeError(f"no probe position header in {path}")

def read_son(subdir, case_name, sonde):
    """(time, values, (x_eff, y_eff)) of probe `sonde` in a build subdir."""
    path = f"{BUILD}/{subdir}/{case_name}_{sonde}.son"
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"no probe entry found in {path}"
    t, v = son.getValues(entries[0])
    return np.asarray(t, float), np.asarray(v, float), probe_xy(path)

print("BUILD =", BUILD)


In [ ]:
from trustutils import run

# Sequential (not in parallel_ok): nbProcs = 1, no partition.
NX0, NY0 = 21, 11
meshes = {
    "coarse":   {"nx": NX0, "ny": NY0},
    "standard": {"nx": 2*NX0-1, "ny": 2*NY0-1},
    "fine":     {"nx": 4*NX0-3, "ny": 4*NY0-3},
}
cases = {}
for lvl, dic in meshes.items():
    cases[lvl] = run.addCaseFromTemplate("hartmann_developing.data", lvl, dic=dic, nbProcs=1)
run.printCases()
run.runCases()


In [ ]:
Ha, h, mu = 5.0, 0.5, 1.0
U_INLET = 0.1  # imposed plug velocity (mass flow per unit depth M = U_INLET * 1.0)

results = {}
for lvl in ("coarse", "standard", "fine"):
    cname = cases[lvl].dataFileName_
    # centerline development probes
    ts = []
    us = []
    for i in range(5):
        t_i, u_i, (x_i, _) = read_son(lvl, cname, f"SONDE_U{i}")
        ts.append(np.asarray(t_i, float)); us.append(float(np.asarray(u_i, float)[-1]))
    t_in, u_in0 = ts[0], us[0]
    # outlet profile probes
    prof_ys = [-0.4, -0.2, 0.0, 0.2, 0.4]
    prof_names = ["SONDE_YM2", "SONDE_YM1", "SONDE_UB", "SONDE_YP1", "SONDE_YP2"]
    prof_u = []
    for name in prof_names:
        t_p, u_p, (x_p, y_p) = read_son(lvl, cname, name)
        prof_u.append(float(np.asarray(u_p, float)[-1]))
    # Hartmann reference from the (near-plug) inlet mass flow
    M = 2*h*us[0]
    G = M*mu*Ha*Ha/(2*h**3*(1.0-np.tanh(Ha)/Ha))
    u_h = G*h*h/(mu*Ha*Ha)*(1.0-1.0/np.cosh(Ha))
    u_out = us[-1]
    dev = (u_out - U_INLET)/(u_h - U_INLET) if abs(u_h-U_INLET) > 1e-12 else float("nan")
    results[lvl] = dict(us=us, prof_ys=prof_ys, prof_u=prof_u, u_out=u_out,
                        u_h=u_h, dev=dev, M=M)
    print(f"{lvl:>8}: u_in(probe0)={us[0]:.6g} | outlet centerline={u_out:.6g} | "
          f"Hartmann={u_h:.6g} | development={dev:.3f}")

# --- Tighter physical checks ---
# 1. Monotonic development along the duct (centerline velocity increases downstream)
for lvl in ("coarse", "standard", "fine"):
    us = results[lvl]["us"]
    for i in range(len(us)-1):
        assert us[i+1] >= us[i] - 1e-3, \
            f"{lvl}: non-monotonic development (u{i}={us[i]:.4g} > u{i+1}={us[i+1]:.4g})"
print("OK: centerline velocity increases monotonically downstream.")

# 2. Outlet centerline vs fully-developed Hartmann limit (diagnostic)
for lvl in ("coarse", "standard", "fine"):
    r = results[lvl]
    flag = "OK" if r["u_out"] <= r["u_h"] + 1e-9 else "ABOVE LIMIT (developing/overshoot)"
    print(f"{lvl:>8}: outlet={r['u_out']:.4g}  Hartmann={r['u_h']:.4g}  -> {flag}")

# 3. Mass-flow conservation at the outlet (trapezoidal integration of profile)
#    Inlet mass flow per unit depth M = U_INLET * height = 0.1
for lvl in ("coarse", "standard", "fine"):
    r = results[lvl]
    ys = [-0.5] + r["prof_ys"] + [0.5]
    us_p = [0.0] + r["prof_u"] + [0.0]
    M_out = float(np.trapezoid(us_p, ys))
    M_in = U_INLET * 1.0
    err = abs(M_out - M_in)
    print(f"{lvl:>8}: M_in={M_in:.4g}  M_out(trapezoid)={M_out:.4g}  |err|={err:.3e}")
    assert err < 0.05, f"{lvl}: mass flow not conserved (err={err:.3e})"
print("OK: outlet mass flow conserved to within 5e-2.")

# 4. Profile shape: centerline is the maximum (Hartmann-like)
# Effective probe positions come from the .son header (grav snaps to cell
# centres, so nominal y and effective y differ, especially on coarse meshes).
for lvl in ("coarse", "standard", "fine"):
    cname = cases[lvl].dataFileName_
    r = results[lvl]
    pu = r["prof_u"]
    # read effective y positions
    eff_ys = []
    for name in prof_names:
        _, _, (_, yeff) = read_son(lvl, cname, name)
        eff_ys.append(yeff)
    print(f"{lvl:>8}: y_eff = " + "  ".join(f"{y:.3f}" for y in eff_ys))
    print(f"{lvl:>8}: u      = " + "  ".join(f"{v:.4g}" for v in pu))
    # centerline (index 2) is the max
    assert pu[2] >= max(pu[0], pu[1], pu[3], pu[4]) - 1e-9, \
        f"{lvl}: centerline is not the velocity maximum"
    # symmetry diagnostic (probe snapping makes exact symmetry mesh-dependent)
    d1 = abs(pu[1]-pu[3]); d2 = abs(pu[0]-pu[4])
    print(f"{lvl:>8}: symmetry diffs: |u(-.2)-u(.2)|={d1:.3e}  |u(-.4)-u(.4)|={d2:.3e}")
print("OK: outlet profile has centerline maximum (Hartmann-like).")

# 5. Full Hartmann profile shape comparison (normalized, reference-independent)
# Fit the Hartmann centerline to the observed mass flow and compare the
# normalized profile shape (removes dependence on the uncertain mass flow).
for lvl in ("coarse", "standard", "fine"):
    cname = cases[lvl].dataFileName_
    r = results[lvl]
    pu = np.asarray(r["prof_u"], float)
    # effective y positions from .son headers
    eff_ys = []
    for name in prof_names:
        _, _, (_, yeff) = read_son(lvl, cname, name)
        eff_ys.append(yeff)
    eff_ys = np.asarray(eff_ys, float)
    # Hartmann shape (normalized by centerline) using observed mass flow
    M_obs = float(np.trapezoid([0.0]+list(pu)+[0.0], [-0.5]+list(eff_ys)+[0.5]))
    U_avg = M_obs / 1.0
    # centerline for this mass flow
    u_h_fit = U_avg * (1.0 - 1.0/np.cosh(Ha)) / (1.0 - np.tanh(Ha)/Ha)
    # Hartmann profile at probe y
    hart = u_h_fit * (1.0 - np.cosh(Ha*eff_ys/h)/np.cosh(Ha)) / (1.0 - 1.0/np.cosh(Ha))
    # normalized shapes
    pu_n = pu / pu[2] if pu[2] != 0 else pu
    hart_n = hart / hart[2] if hart[2] != 0 else hart
    shape_err = float(np.max(np.abs(pu_n - hart_n))) if len(pu_n)==len(hart_n) else float("nan")
    print(f"{lvl:>8}: mass-flow-fitted Hartmann centerline={u_h_fit:.4g}  "
          f"max normalized shape error={shape_err:.3e}")
    results[lvl]["shape_err"] = shape_err
# Shape should improve with refinement (loose bound)
assert results["fine"]["shape_err"] < 0.5, \
    f"profile shape does not resemble Hartmann: {results['fine']['shape_err']:.3e}"
print("OK: outlet profile resembles Hartmann shape (normalized).")

# 6. Time convergence: outlet centerline must be quasi-steady by tmax
for lvl in ("coarse", "standard", "fine"):
    cname = cases[lvl].dataFileName_
    t, v, _ = read_son(lvl, cname, "SONDE_U4")
    t = np.asarray(t, float); v = np.asarray(v, float)
    tail = v[-5:]
    drift = float(np.max(tail) - np.min(tail))
    print(f"{lvl:>8}: last-5-step drift in outlet u = {drift:.3e}")
    assert drift < 1e-3, f"{lvl}: not time-converged (drift={drift:.3e})"
print("OK: outlet velocity is time-converged (drift < 1e-3 over last 5 steps).")

# 5. Mesh convergence of the outlet centreline velocity
u_outs = [results[l]["u_out"] for l in ("coarse","standard","fine")]
conv = abs(u_outs[2]-u_outs[1])
print(f"\noutlet u convergence (standard->fine): {conv:.3e}")
assert conv < 5e-2, f"outlet velocity not converged: {conv:.3e}"
print("OK: mesh-converged developing flow (outlet u stable to 5e-2).")


In [ ]:
run.tablePerf()
